### Question - 007: - Write a pyspark query using below input data to get provided output

Input Data : 
|Product|Region|Sales|
|-------|------|-----|
|      A|  East|  100|
|      A|  West|  150|
|      A|  West|  150|
|      B|  East|  200|
|      B|  West|  250|
|      C|  East|  300|
|      C|  West|  350|
|      C|  West|  150|

Output Data : `Pivot Table` 
|Product|East|West|
|-------|----|----|
|      B| 200| 250|
|      C| 300| 500|
|      A| 100| 300|


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-q-007")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark.sparkContext.setLogLevel("ERROR")

#
# 
#
def display(df):
    df.show(truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 10:24:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://vmware-kubuntu.sandbox.net:4040


In [2]:
# Sample data
data = [
    ("A", "East", 100),
    ("A", "West", 150),
    ("A", "West", 150),
    ("B", "East", 200),
    ("B", "West", 250),
    ("C", "East", 300),
    ("C", "West", 350),
    ("C", "West", 150)
]

# Create a DataFrame
columns = ["Product", "Region", "Sales"]
df = spark.createDataFrame(data, columns)

# Show the initial DataFrame
df.show()

+-------+------+-----+
|Product|Region|Sales|
+-------+------+-----+
|      A|  East|  100|
|      A|  West|  150|
|      A|  West|  150|
|      B|  East|  200|
|      B|  West|  250|
|      C|  East|  300|
|      C|  West|  350|
|      C|  West|  150|
+-------+------+-----+



In [3]:
from pyspark.sql.functions import *
# ...

# Pivot the DataFrame
pivot_df = df.groupBy("Product").pivot("Region").sum("Sales")

# Show the pivoted DataFrame
pivot_df.show()

+-------+----+----+
|Product|East|West|
+-------+----+----+
|      B| 200| 250|
|      C| 300| 500|
|      A| 100| 300|
+-------+----+----+



In [8]:
# Pivot and sum the sales
pivot_df_with_aggregation_sum = df.groupBy("Product").pivot("Region").sum("Sales")

# Show the result
pivot_df_with_aggregation_sum.show()



+-------+----+----+
|Product|East|West|
+-------+----+----+
|      B| 200| 250|
|      C| 300| 500|
|      A| 100| 300|
+-------+----+----+



In [9]:
# Pivot and sum the sales
pivot_df_with_aggregation_avg = df.groupBy("Product").pivot("Region").avg("Sales")

# Show the result
pivot_df_with_aggregation_avg.show()

+-------+-----+-----+
|Product| East| West|
+-------+-----+-----+
|      B|200.0|250.0|
|      C|300.0|250.0|
|      A|100.0|150.0|
+-------+-----+-----+



#### Un Pivot 

In [7]:

# Applying unpivot()
from pyspark.sql.functions import expr

unpivotExpr = "stack(3, 'East', East, 'West', West) as (Region,Sales)"
unPivotDF = pivot_df.select("Product", expr(unpivotExpr)) \
    .where("Sales is not null")
unPivotDF.show(truncate=False)


+-------+------+-----+
|Product|Region|Sales|
+-------+------+-----+
|B      |East  |200  |
|B      |West  |250  |
|C      |East  |300  |
|C      |West  |500  |
|A      |East  |100  |
|A      |West  |300  |
+-------+------+-----+

